# PyDI Data Integration Workflow: Companies

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with companies datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
- [Part 2: Data Profiling](#part-2-data-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [35]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [36]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

## Step 1: Load Target Schema and Normalization Spec

In [37]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")
spec.set_column("founders", output_type="list<string>")
spec.set_column("founded", output_type="string")
target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,website,string
3,founded,string
4,country,string
5,city,string
6,industry,string
7,assets,int
8,revenue,int
9,founders,list<string>


## Step 2: Load Source Datasets

In [38]:
from PyDI.io import load_csv, load_xml, load_json
import numpy as np


def normalize_iso_date_column(df, column):
    if column not in df.columns:
        return df

    original = df[column]
    text = original.astype("string").str.strip()
    missing = original.isna() | text.eq("")
    parsed = pd.to_datetime(text, errors="coerce")
    normalized = parsed.dt.strftime("%Y-%m-%d")
    df[column] = normalized.where(parsed.notna(), original)
    df.loc[missing, column] = pd.NA
    return df


dbpedia = load_csv(INPUT_DIR / "data" / "dbpedia.csv")
dbpedia.attrs["dataset_name"] = "dbpedia"
normalize_iso_date_column(dbpedia, "established")
dbpedia["keypeople_name"] = dbpedia["keypeople_name"].apply(lambda x: x if x else np.nan)
dbpedia.head(10)

,entity_uri,org_name,established,nation,headquarters,sector,keypeople_name,total_assets_val,annual_income
0,http://dbpedia.org/resource/%C3%80_la_Table_de...,À la Table de Spanghero,1970-01-01,France,Castelnaudary,Meat,NaN,NaN,NaN
1,http://dbpedia.org/resource/%C3%81guas_de_Port...,�?guas de Portugal,1993-01-01,Portugal,Lisbon,NaN,NaN,NaN,NaN
2,http://dbpedia.org/resource/%C3%81nima_Estudios,�?nima Estudios,2002-01-01,Mexico,Mexico City,Animation,NaN,NaN,NaN
3,http://dbpedia.org/resource/%C3%87al%C4%B1k_En...,Çalık Enerji,1998-01-01,Turkey,Istanbul,NaN,Çalık Holding,NaN,NaN
4,http://dbpedia.org/resource/%C3%87al%C4%B1k_Ho...,Çalık Holding,1997-01-01,Turkey,Istanbul,NaN,Ahmet Çalık,8.000000e+00,2.800000e+00
5,http://dbpedia.org/resource/%C3%87ukurova_(con...,Çukurova (construction firm),1975-01-01,Turkey,Istanbul,NaN,NaN,NaN,NaN
6,http://dbpedia.org/resource/%C3%87ukurova_Holding,Çukurova Holding,1923-01-01,Turkey,Istanbul,Communication,NaN,NaN,NaN
7,http://dbpedia.org/resource/%C3%89lectricit%C3...,Électricité de France,1946-01-01,France,Paris,Electric utility,Marcel Paul,2.405600e+11,6.517000e+10
8,http://dbpedia.org/resource/%C3%89tranges_Libe...,Étranges Libellules,1994-01-01,France,Lyon,Video game industry,NaN,NaN,NaN
9,http://dbpedia.org/resource/%C3%96ssur,Össur,1971-01-01,Iceland,Reykjavík,Health care,NaN,6.071000e+08,3.585000e+08


In [39]:
forbes = load_csv(INPUT_DIR / "data" / "forbes.csv")
forbes.attrs["dataset_name"] = "forbes"
forbes.head()

,forbes_url,company,url,region,business_segment,asset_value,sales_figure
0,http://www.forbes.com/companies/icbc/,ICBC,http://www.forbes.com/companies/icbc/,China,Major Banks,3124900000000,148700000000
1,http://www.forbes.com/companies/china-construc...,China Construction Bank,http://www.forbes.com/companies/china-construc...,China,Regional Banks,2449500000000,121300000000
2,http://www.forbes.com/companies/agricultural-b...,Agricultural Bank of China,http://www.forbes.com/companies/agricultural-b...,China,Regional Banks,2405400000000,136400000000
3,http://www.forbes.com/companies/jpmorgan-chase/,JPMorgan Chase,http://www.forbes.com/companies/jpmorgan-chase/,United States of America,Major Banks,2435300000000,105700000000
4,http://www.forbes.com/companies/berkshire-hath...,Berkshire Hathaway,http://www.forbes.com/companies/berkshire-hath...,United States of America,Investment Services,493400000000,178800000000


In [40]:
fullcontact = load_csv(INPUT_DIR / "data" / "fullcontact.csv")
fullcontact.attrs["dataset_name"] = "fullcontact"
normalize_iso_date_column(fullcontact, "Attribute_6")
fullcontact.head()

,Attribute_1,Attribute_2,Attribute_3,Attribute_4,Attribute_5,Attribute_6
0,fullcontact_1,BBMG,United States,Brooklyn,Raphael Bemporad,<NA>
1,fullcontact_2,CIT Group Inc (DEL),Canada,Toronto,NaN,1908-01-01
2,fullcontact_3,City & National Employment,United States,Waterloo,NaN,1957-01-01
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,Ireland,Swords,NaN,1871-01-01
4,fullcontact_5,Evonik,Germany,Essen,NaN,2007-01-01


In [41]:
# Create founders column based on keypersons_person_name and keypersons_person_title
# Ensure name/title columns are always list-like or NaN
fullcontact["founders"] = fullcontact["Attribute_5"].apply(
    lambda x: [x] if isinstance(x, str) else x
)


## Step 3: LLM-Based Schema Matching

In [42]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match dbpedia dataset
dbpedia_mapping = matcher.match(dbpedia, df_target)

dbpedia_mapping

[INFO ] PyDI.schemamatching.llm_based - Initialized LLMBasedSchemaMatcher with 40 sample rows
[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: dbpedia -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2026-06-02T10:30:21.615834Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 14640.694856643677, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 4181, "output_tokens": 1170, "total_tokens": 5351, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 1088}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nIMPORTANT: Source column names may be placeholders or generic identifiers (for example Attribute_1, col1, field_a, Column_A). When column names are not descriptive, focus on the actual data valu

,source_dataset,source_column,target_dataset,target_column,score,notes
0,dbpedia,entity_uri,target_schema,id,0.95,llm_based_matching
1,dbpedia,org_name,target_schema,name,0.95,llm_based_matching
2,dbpedia,established,target_schema,founded,0.95,llm_based_matching
3,dbpedia,nation,target_schema,country,0.95,llm_based_matching
4,dbpedia,headquarters,target_schema,city,0.95,llm_based_matching
5,dbpedia,sector,target_schema,industry,0.95,llm_based_matching
6,dbpedia,keypeople_name,target_schema,founders,0.95,llm_based_matching
7,dbpedia,total_assets_val,target_schema,assets,0.95,llm_based_matching
8,dbpedia,annual_income,target_schema,revenue,0.95,llm_based_matching


In [43]:
forbes_mapping = matcher.match(forbes, df_target)
# add forbes_url as id to the mapping

forbes_mapping = pd.concat([
    forbes_mapping,
    pd.DataFrame([{
        "source_dataset": "forbes",
        "source_column": "forbes_url",
        "target_dataset": "target_schema",
        "target_column": "id",
        "score": 1.0,
        "notes": "manual",
    }]),
], ignore_index=True)

forbes_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: forbes -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2026-06-02T10:30:40.143985Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 18500.64706802368, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 3745, "output_tokens": 1343, "total_tokens": 5088, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 1280}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nIMPORTANT: Source column names may be placeholders or generic identifiers (for example Attribute_1, col1, field_a, Column_A). When column names are not descriptive, focus on the actual data values to infer each column's semantic meaning:\n- Look at value formats, patterns, and content in t

,source_dataset,source_column,target_dataset,target_column,score,notes
0,forbes,forbes_url,target_schema,id,0.95,llm_based_matching
1,forbes,company,target_schema,name,0.95,llm_based_matching
2,forbes,region,target_schema,country,0.95,llm_based_matching
3,forbes,business_segment,target_schema,industry,0.95,llm_based_matching
4,forbes,asset_value,target_schema,assets,0.95,llm_based_matching
5,forbes,sales_figure,target_schema,revenue,0.95,llm_based_matching
6,forbes,forbes_url,target_schema,id,1.00,manual


In [44]:
fullcontact_mapping = matcher.match(fullcontact, df_target)
fullcontact_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: fullcontact -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2026-06-02T10:31:01.251390Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 21080.241203308105, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 3378, "output_tokens": 1417, "total_tokens": 4795, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 1344}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nIMPORTANT: Source column names may be placeholders or generic identifiers (for example Attribute_1, col1, field_a, Column_A). When column names are not descriptive, focus on the actual data values to infer each column's semantic meaning:\n- Look at value formats, patterns, and conten

,source_dataset,source_column,target_dataset,target_column,score,notes
0,fullcontact,Attribute_1,target_schema,id,0.95,llm_based_matching
1,fullcontact,Attribute_2,target_schema,name,0.95,llm_based_matching
2,fullcontact,Attribute_3,target_schema,country,0.95,llm_based_matching
3,fullcontact,Attribute_4,target_schema,city,0.95,llm_based_matching
4,fullcontact,Attribute_5,target_schema,founders,0.95,llm_based_matching
5,fullcontact,Attribute_6,target_schema,founded,0.95,llm_based_matching
6,fullcontact,founders,target_schema,founders,0.95,llm_based_matching


## Step 4: Translate and Normalize


In [45]:
translator = SchemaTranslator()
# Translate + normalize each dataset with its own mapping

# Adjust forbes financials from billions to absolute numbers
#forbes["sales_figure"] = forbes["sales_figure"].apply(lambda x: x * 1e9 if pd.notna(x) else x)
#forbes["asset_value"] = forbes["asset_value"].apply(lambda x: x * 1e9 if pd.notna(x) else x)

# Remove , from dbpedia financials
dbpedia["total_assets_val"] = dbpedia["total_assets_val"].astype(str).str.replace(",", "", regex=False)
dbpedia["annual_income"] = dbpedia["annual_income"].astype(str).str.replace(",", "", regex=False)
for col in ["total_assets_val", "annual_income"]:
    dbpedia[col] = pd.to_numeric(dbpedia[col], errors="coerce")
    dbpedia[col] = dbpedia[col].round().astype("Int64")

spec.set_column("country", country_format="name")

forbes_normalized = translator.translate(
    forbes, forbes_mapping,
    normalize=spec, on_failure="keep"
)

dbpedia_normalized = translator.translate(
    dbpedia, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

fullcontact_normalized = translator.translate(
    fullcontact, fullcontact_mapping,
    normalize=spec, on_failure="keep"
)

# =========================================================================
# 3. Text structural feature isolation for model matching
# =========================================================================
import re
import unicodedata

STRUCTURAL_CLEAN_MAP = {
    "texasdallas": "dallas texas",
    "californiamilpitas california": "milpitas california",
    "texashouston": "houston texas",
    "korea republic of": "south korea",
    "republic of korea": "south korea",
    "peoples republic of china": "china",
    "hong kong china": "hong kong",
    "new york city": "new york",
    "midtown manhattan": "new york",
    "winstonsalem north california": "winstonsalem",
    "fujianlongyanshanghang": "shanghang longyan fujian"
}

def clean_structural_anomalies(text):
    if not text or pd.isna(text):
        return ""
    
    s = str(text).strip().lower()
    s = re.sub(r"\(.*?\)", "", s) # Drop trailing descriptions like (city)
    s = s.split(';')[0].strip()   # Capture text prior to semi-colon lists
    
    # Dynamic accent / diacritic stripping
    s = "".join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    
    # Strip layout punctuation and commas
    s = re.sub(r"[^\w\s]", "", s)  
    s = re.sub(r"\s+", " ", s).strip()
    
    if s.endswith(" city"):
        s = s[:-5]
    if " bang rak district" in s:
        s = s.replace(" bang rak district", "").replace("si lom ", "")
        
    if s in STRUCTURAL_CLEAN_MAP:
        return STRUCTURAL_CLEAN_MAP[s]
        
    return s

# =========================================================================
# 4. Apply the structural cleaning function to the normalized datasets
# =========================================================================
# We iterate through the normalized datasets and apply the function
# We also use .get() to handle cases where 'city' or 'country' might be missing
for df in [forbes_normalized, dbpedia_normalized, fullcontact_normalized]:
    # Ensure columns exist before trying to apply (avoid KeyError)
    if 'city' in df.columns:
        df['match_city'] = df['city'].apply(clean_structural_anomalies)
    else:
        df['match_city'] = ""
        
    if 'country' in df.columns:
        df['match_country'] = df['country'].apply(clean_structural_anomalies)
    else:
        df['match_country'] = ""

# Force-fix DBpedia financials to actual numbers
# Using the TARGET schema names ('assets' and 'revenue') that the translator created
for df in [dbpedia_normalized]:
    for col in ['assets', 'revenue']: 
        if col in df.columns:
            # Convert to numeric, then to Int64
            df[col] = pd.to_numeric(df[col], errors='coerce').round().astype('Int64')

[INFO ] root - Translating 6 columns for 'forbes'
[INFO ] PyDI.normalization.taxonomy - Loaded 74 taxonomy values from 'Industry Name'
[INFO ] PyDI.normalization.transform - Taxonomy mapping for forbes.industry: 80 unique, 9 exact, 0 cached, 71 need LLM
[INFO ] PyDI.normalization.transform - Column 'industry' has 71 values outside the taxonomy and no chat_model was provided; leaving those values unchanged
[WARNING] PyDI.normalization.transform - Column 'website' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'founded' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'city' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'founders' not found in DataFrame
[INFO ] root - Normalization complete: 2038 values transformed, 1848 values failed
[INFO ] root - Translating 9 columns for 'dbpedia'
[INFO ] PyDI.normalization.taxonomy - Loaded 74 taxonomy values from 'Industry Name'
[INFO ] PyDI.normalization.transform - Taxonomy ma

In [46]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head()

,id,name,founded,country,city,industry,assets,revenue,founders
0,http://dbpedia.org/resource/%C3%80_la_Table_de...,À la Table de Spanghero,1970-01-01,France,Castelnaudary,Meat,<NA>,<NA>,NaN
1,http://dbpedia.org/resource/%C3%81guas_de_Port...,�?guas de Portugal,1993-01-01,Portugal,Lisbon,NaN,<NA>,<NA>,NaN
2,http://dbpedia.org/resource/%C3%81nima_Estudios,�?nima Estudios,2002-01-01,Mexico,Mexico City,Animation,<NA>,<NA>,NaN
3,http://dbpedia.org/resource/%C3%87al%C4%B1k_En...,Çalık Enerji,1998-01-01,Turkey,Istanbul,NaN,<NA>,<NA>,Çalık Holding
4,http://dbpedia.org/resource/%C3%87al%C4%B1k_Ho...,Çalık Holding,1997-01-01,Turkey,Istanbul,NaN,8,3,Ahmet Çalık


In [47]:
# Inspect normalized forbes dataset (target columns only)
forbes_cols = [c for c in target_columns if c in forbes_normalized.columns]
forbes_normalized[forbes_cols].head()

,id,name,country,industry,assets,revenue
0,http://www.forbes.com/companies/icbc/,ICBC,China,Major Banks,3124900000000,148700000000
1,http://www.forbes.com/companies/china-construc...,China Construction Bank,China,Regional Banks,2449500000000,121300000000
2,http://www.forbes.com/companies/agricultural-b...,Agricultural Bank of China,China,Regional Banks,2405400000000,136400000000
3,http://www.forbes.com/companies/jpmorgan-chase/,JPMorgan Chase,United States,Major Banks,2435300000000,105700000000
4,http://www.forbes.com/companies/berkshire-hath...,Berkshire Hathaway,United States,Investment Services,493400000000,178800000000


In [48]:
# Inspect normalized fullcontact dataset (target columns only)
fullcontact_cols = [c for c in target_columns if c in fullcontact_normalized.columns]
fullcontact_normalized[fullcontact_cols].head()

,id,name,founded,country,city,founders
0,fullcontact_1,BBMG,<NA>,United States,Brooklyn,Raphael Bemporad
1,fullcontact_2,CIT Group Inc (DEL),1908-01-01,Canada,Toronto,NaN
2,fullcontact_3,City & National Employment,1957-01-01,United States,Waterloo,NaN
3,fullcontact_4,Ingersoll Rand South East Asia (Pte) Ltd,1871-01-01,Ireland,Swords,NaN
4,fullcontact_5,Evonik,2007-01-01,Germany,Essen,NaN


In [49]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
forbes = forbes_normalized[forbes_cols].copy()
fullcontact = fullcontact_normalized[fullcontact_cols].copy()

## Part 2: Data Profiling

In [50]:
from PyDI.utils import DataProfiler

# Display basic information
datasets = [dbpedia, forbes, fullcontact]
names = ["DBpedia", "Forbes", "FullContact"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

Total records across all datasets: 14,016
dbpedia:
  Rows: 10,085
  Columns: 9
  Total nulls: 30,513
  Null percentage: 33.6%
  Null counts per column:
    city: 700 (6.9%)
    industry: 3,207 (31.8%)
    assets: 9,425 (93.5%)
    revenue: 8,024 (79.6%)
    founders: 9,157 (90.8%)

forbes:
  Rows: 2,000
  Columns: 6
  Total nulls: 114
  Null percentage: 0.9%
  Null counts per column:
    country: 71 (3.5%)
    industry: 43 (2.1%)

fullcontact:
  Rows: 1,931
  Columns: 6
  Total nulls: 3,678
  Null percentage: 31.7%
  Null counts per column:
    founded: 875 (45.3%)
    country: 508 (26.3%)
    city: 556 (28.8%)
    founders: 1,739 (90.1%)



{'rows': 1931,
 'columns': 6,
 'nulls_total': 3678,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'founded': 875,
  'country': 508,
  'city': 556,
  'founders': 1739},
 'dtypes': {'id': 'object',
  'name': 'object',
  'founded': 'object',
  'country': 'object',
  'city': 'object',
  'founders': 'object'}}

### Attribute Coverage Analysis

In [51]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\nAttributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

[INFO ] PyDI.fusion.analysis - Analyzed 9 attributes across 3 datasets


Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,forbes_count,forbes_pct,forbes_coverage,forbes_samples,fullcontact_count,fullcontact_pct,fullcontact_coverage,fullcontact_samples,avg_coverage,max_coverage,datasets_with_attribute
0,assets,660/10085,6.5%,0.065444,"[8, 240560000000, 607100000]",2000/2000,100.0%,1.0000,"[3124900000000, 2449500000000, 2405400000000]",0/0,0%,0.000000,N/A,0.355148,1.00000,2
1,city,9385/10085,93.1%,0.930590,"['Castelnaudary', 'Lisbon', 'Mexico City']",0/0,0%,0.0000,N/A,1375/1931,71.2%,0.712066,"['Brooklyn', 'Toronto', 'Waterloo']",0.547552,0.93059,2
2,country,10085/10085,100.0%,1.000000,"['France', 'Portugal', 'Mexico']",1929/2000,96.5%,0.9645,"['China', 'China', 'China']",1423/1931,73.7%,0.736924,"['United States', 'Canada', 'United States']",0.900475,1.00000,3
3,founded,10085/10085,100.0%,1.000000,"['1970-01-01', '1993-01-01', '2002-01-01']",0/0,0%,0.0000,N/A,1056/1931,54.7%,0.546867,"['1908-01-01', '1957-01-01', '1871-01-01']",0.515622,1.00000,2
4,founders,928/10085,9.2%,0.092018,"['Çalık Holding', 'Ahmet Çalık', 'Marcel Paul']",0/0,0%,0.0000,N/A,192/1931,9.9%,0.099430,"['Raphael Bemporad', 'John Pitcairn', 'Douglas...",0.063816,0.09943,2
5,id,10085/10085,100.0%,1.000000,['http://dbpedia.org/resource/%C3%80_la_Table_...,2000/2000,100.0%,1.0000,"['http://www.forbes.com/companies/icbc/', 'htt...",1931/1931,100.0%,1.000000,"['fullcontact_1', 'fullcontact_2', 'fullcontac...",1.000000,1.00000,3
6,industry,6878/10085,68.2%,0.682003,"['Meat', 'Animation', 'Communication']",1957/2000,97.9%,0.9785,"['Major Banks', 'Regional Banks', 'Regional Ba...",0/0,0%,0.000000,N/A,0.553501,0.97850,2
7,name,10085/10085,100.0%,1.000000,"['À la Table de Spanghero', '�?guas de Portuga...",2000/2000,100.0%,1.0000,"['ICBC', 'China Construction Bank', 'Agricultu...",1931/1931,100.0%,1.000000,"['BBMG', 'CIT Group Inc (DEL)', 'City & Nation...",1.000000,1.00000,3
8,revenue,2061/10085,20.4%,0.204363,"[3, 65170000000, 358500000]",2000/2000,100.0%,1.0000,"[148700000000, 121300000000, 136400000000]",0/0,0%,0.000000,N/A,0.401454,1.00000,2



Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['assets', 'city', 'country', 'founded', 'founders', 'id', 'industry', 'name', 'revenue']


## Part 3: Entity Matching

### Step 1: Blocking

In [52]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [53]:
from PyDI.entitymatching import TokenBlocker

token_blocker_f2d = TokenBlocker(
    forbes_normalized, dbpedia_normalized,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

token_blocker_f2fc = TokenBlocker(
    forbes_normalized, fullcontact_normalized,
    column='name',
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2295 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 11043 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1111 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/blocking-evaluation/debugResultsBlocking_TokenBlocker.csv
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2295 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 2709 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 1122 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/aaronsteiner/Documents/Git

### Step 2: Evaluate Blocking Against Ground Truth

In [54]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 4 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 7 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 9 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 11 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 12 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 12 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 12 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 12 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 14 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 22 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 27 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 42 true matches
[INFO ] root - Processed 130 batches, 129861 pairs, 101 true matches
[INFO ] root -   Pair Completeness: 0.971
[INFO ] root -   Pair Quality:      0.0

{'pair_completeness': 0.9711538461538461,
 'pair_quality': 0.0007777546761537336,
 'reduction_ratio': 0.9935616757560733,
 'total_candidates': 129861,
 'total_possible_pairs': 20170000,
 'true_positives_found': 101,
 'total_true_pairs': 104,
 'batches_processed': 130,
 'evaluation_timestamp': '2026-06-02T12:31:05.745325',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/blocking-evaluation/blocking_detailed_results.csv']}

In [55]:
# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=token_blocker_f2fc,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 12 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 15 true matches
[INFO ] root - Processed 30 batches, 29500 pairs, 218 true matches
[INFO ] root -   Pair Completeness: 0.944
[INFO ] root -   Pair Quality:      0.007
[INFO ] root -   Reduction Ratio:   0.992361
[INFO ] root -   True Matches Found: 218/231
[INFO ] root -   Batches Processed:  30
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9437229437229437,
 'pair_quality': 0.007389830508474577,
 'reduction_ratio': 0.9923614707405489,
 'total_candidates': 29500,
 'total_possible_pairs': 3862000,
 'true_positives_found': 218,
 'total_true_pairs': 231,
 'batches_processed': 30,
 'evaluation_timestamp': '2026-06-02T12:31:07.280374',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [56]:
from PyDI.entitymatching import StringComparator
import re

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators_f2d = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='name', 
        similarity_function='levenshtein',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='industry',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

comparators_f2fc = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [57]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_f2d = matcher.match(
    df_left=forbes_normalized,
    df_right=dbpedia_normalized, 
    candidates=token_blocker_f2d,
    comparators=comparators_f2d,
    weights=[1.0, 1.0, 1.0, 0.3],
    threshold=0.2,
    id_column='id'
)

correspondences_f2fc = matcher.match(
    df_left=forbes_normalized,
    df_right=fullcontact_normalized, 
    candidates=token_blocker_f2fc,
    comparators=comparators_f2fc,
    weights=[1.0, 0.5],
    threshold=0.1,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 10085 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 10085 elements after 0:00:0.039; 129861 blocked pairs (reduction ratio: 0.9935616757560733)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:13.816; found 59240 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 1931 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 1931 elements after 0:00:0.012; 29500 blocked pairs (reduction ratio: 0.9923614707405489)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:2.350; found 27073 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [58]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  101
[INFO ] root -   True Negatives:  60
[INFO ] root -   False Positives: 55
[INFO ] root -   False Negatives: 3
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.735
[INFO ] root -   Precision: 0.647
[INFO ] root -   Recall:    0.971
[INFO ] root -   F1-Score:  0.777


{'precision': 0.6474358974358975,
 'recall': 0.9711538461538461,
 'f1': 0.7769230769230769,
 'accuracy': 0.7351598173515982,
 'true_positives': 101,
 'false_positives': 55,
 'false_negatives': 3,
 'true_negatives': 60,
 'threshold_used': 0.0,
 'total_correspondences': 59240,
 'filtered_correspondences': 59240,
 'evaluation_timestamp': '2026-06-02T12:31:24.406553',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [59]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n Cluster Size Distribution Results:")
display(cluster_distribution)

Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 184 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	145	|	78.80%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	18	|	9.78%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	8	|	4.35%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	3	|	1.63%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	2	|	1.09%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	4	|	2.17%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		22	|	1	|	0.54%
[INFO ] PyDI.entitymatching.evaluation - 		5347	|	1	|	0.54%
[INFO ] root - Cluster size distribution written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/cluster_analysis/cluster_size_distribution.c


 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,145,78.804348
1,3,18,9.782609
2,4,8,4.347826
3,5,3,1.630435
4,6,2,1.086957
5,7,4,2.173913
6,8,1,0.543478
7,11,1,0.543478
8,22,1,0.543478
9,5347,1,0.543478


In [60]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_f2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 184 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [61]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm

# use Greedy One-To-One Matching to refine results to 1:1 matches
clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2d = clusterer.cluster(correspondences_f2d)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] root - Filtered correspondences: 59240 -> 59240 (threshold=0.0)
[INFO ] root - Greedy matching: 59240 -> 1404 correspondences (2808 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 59240 -> 1404 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 5819 -> 2808 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  93
[INFO ] root -   True Negatives:  109
[INFO ] root -   False Positives: 6
[INFO ] root -   False Negatives: 11
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.922
[INFO ] root -   Precision: 0.939
[INFO ] root -   Recall:    0.894
[INFO ] root -   F1-Score:  0.916


{'precision': 0.9393939393939394,
 'recall': 0.8942307692307693,
 'f1': 0.9162561576354681,
 'accuracy': 0.9223744292237442,
 'true_positives': 93,
 'false_positives': 6,
 'false_negatives': 11,
 'true_negatives': 109,
 'threshold_used': 0.0,
 'total_correspondences': 1404,
 'filtered_correspondences': 1404,
 'evaluation_timestamp': '2026-06-02T12:31:29.072321',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/debug_results_entity_matching/matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 1404 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	1404	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/cluster_analysis/cluster_size_distribution.csv



 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,1404,100.0


In [62]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2fc = clusterer.cluster(correspondences_f2fc)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  217
[INFO ] root -   True Negatives:  406
[INFO ] root -   False Positives: 92
[INFO ] root -   False Negatives: 14
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.855
[INFO ] root -   Precision: 0.702
[INFO ] root -   Recall:    0.939
[INFO ] root -   F1-Score:  0.804
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 285 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	266	|	93.33%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	12	|	4.21%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	3	|	1.05%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.35%
[INFO ] PyDI.entitymatching.evaluation - 

Now, we can directly use the trained model with PyDIs MLBasedMatcher

## Part 4: Data Fusion

In [63]:
forbes_normalized["forbes_id"] = forbes_normalized["id"]

# Assign trust scores to datasets
forbes_normalized.attrs["trust_score"] = 1
dbpedia_normalized.attrs["trust_score"] = 3
fullcontact_normalized.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_f2d, correspondences_f2fc], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 2,499


## Step 1: Define Fusion Strategy

In [64]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, voting, maximum, most_recent

strategy = DataFusionStrategy('company_fusion_strategy')

strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('assets', prefer_higher_trust)
strategy.add_attribute_fuser('revenue', prefer_higher_trust)
strategy.add_attribute_fuser('founders', union)
strategy.add_attribute_fuser('founded', prefer_higher_trust)
strategy.add_attribute_fuser('country', voting)
strategy.add_attribute_fuser('city', shortest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'assets' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'revenue' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founders' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founded' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'country' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'city' using rule 'shortest_string'


## Step 2: Run Fusion

In [65]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[forbes_normalized, dbpedia_normalized, fullcontact_normalized],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'company_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 4236 of 4236 unique IDs
[INFO ] PyDI.fusion.engine - Created 11517 record groups from 2499 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 11517 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	975	|	8.47%
[INFO ] PyDI.fusion.engine - 		3	|	762	|	6.62%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     assets: 0.85
[INFO ] PyDI.fusion.engine - 

Fused rows: 1,737


,_id,_fusion_sources,_fusion_source_datasets,assets,city,country,forbes_id,founded,founders,id,industry,match_city,match_country,name,revenue,url,_fusion_confidence,_fusion_metadata
0,fullcontact_408,"[http://www.forbes.com/companies/icbc/, fullco...","[forbes, fullcontact]",3124900000000,North Vancouver,China,http://www.forbes.com/companies/icbc/,1973-01-01,None,fullcontact_408,Major Banks,north vancouver,canada,ICBC,148700000000,http://www.forbes.com/companies/icbc/,0.666667,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
1,http://dbpedia.org/resource/China_Construction...,[http://www.forbes.com/companies/china-constru...,"[forbes, dbpedia]",2449500000000,Beijing,China,http://www.forbes.com/companies/china-construc...,1954-01-01,None,http://dbpedia.org/resource/China_Construction...,Investment,beijing,china,China Construction Bank,121300000000,http://www.forbes.com/companies/china-construc...,0.708333,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
2,http://dbpedia.org/resource/Industrial_and_Com...,[http://www.forbes.com/companies/agricultural-...,"[forbes, dbpedia]",2405400000000,Beijing,China,http://www.forbes.com/companies/agricultural-b...,1984-01-01,None,http://dbpedia.org/resource/Industrial_and_Com...,Investment,beijing,china,Agricultural Bank of China,136400000000,http://www.forbes.com/companies/agricultural-b...,0.666667,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
3,http://dbpedia.org/resource/Chase_Aircraft,[http://www.forbes.com/companies/jpmorgan-chas...,"[forbes, dbpedia]",2435300000000,"New JerseyTrenton, New Jersey",United States,http://www.forbes.com/companies/jpmorgan-chase/,1943-01-01,[Michael Stroukoff],http://dbpedia.org/resource/Chase_Aircraft,Aerospace manufacturer,new jerseytrenton new jersey,united states,JPMorgan Chase,105700000000,http://www.forbes.com/companies/jpmorgan-chase/,0.750000,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
4,http://dbpedia.org/resource/Berkshire_Hathaway,[http://www.forbes.com/companies/berkshire-hat...,"[forbes, dbpedia]",484931000000,"Kiewit PlazaOmaha, NebraskaNebraska",United States,http://www.forbes.com/companies/berkshire-hath...,1839-01-01,[Oliver Chace],http://dbpedia.org/resource/Berkshire_Hathaway,Conglomerate (company),kiewit plazaomaha nebraskanebraska,united states,Berkshire Hathaway,178800000000,http://www.forbes.com/companies/berkshire-hath...,0.791667,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


## Step 3: Evaluate Data Fusion

In [66]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("assets", tokenized_match)
strategy.add_evaluation_function("revenue", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("assets", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("founders", set_equality_match)
strategy.add_evaluation_function("founded", year_only_match)
strategy.add_evaluation_function("country", tokenized_match)
strategy.add_evaluation_function("city", tokenized_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'revenue' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founders'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founded'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'city'


In [67]:
from PyDI.fusion import DataFusionEvaluator

fusion_validation_set = load_xml(INPUT_DIR / 'fusion' / 'validation_set_final.xml', name='fusion_validation_set', nested_handling='aggregate')
# rename keypeople_name to founders for evaluation
fusion_validation_set['founders'] = fusion_validation_set['keypeople_name'].apply(lambda x: [x] if isinstance(x, str) else x)

# convert scientific number notation into integers
def convert_scientific_notation(value):
    try:
        if isinstance(value, str) and ('e' in value or 'E' in value):
            return int(float(value))
        return value
    except:
        return value

fusion_validation_set['assets'] = fusion_validation_set['assets'].apply(convert_scientific_notation)
fusion_validation_set['revenue'] = fusion_validation_set['revenue'].apply(convert_scientific_notation) 

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='forbes_id',
    gold_df=fusion_validation_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[WARNING] PyDI.fusion.evaluation - Missing 7 expected/reference records in fused dataset: http://www.forbes.com/companies/accor/, http://www.forbes.com/companies/amp/, http://www.forbes.com/companies/shimano/, http://www.forbes.com/companies/kellogg/, http://www.forbes.com/companies/mtr/, ...
[ERROR] PyDI.fusion.evaluation - year_only_match: could not convert values to date (fused='0020-01-01', expected='1891-01-01T00:00:00')
[ERROR] PyDI.fusion.evaluation - year_only_match: could not convert values to date (fused='1472-01-01', expected='1849-01-01T00:00:00')
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.412 overall accuracy (261/633)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 3

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.412
  macro_accuracy: 0.405
  num_evaluated_records: 93
  num_evaluated_attributes: 7
  total_evaluations: 633
  total_correct: 261
  city_accuracy: 0.344
  city_count: 93
  founders_accuracy: 0.133
  founders_count: 75
  country_accuracy: 0.656
  country_count: 93
  revenue_accuracy: 0.054
  revenue_count: 93
  name_accuracy: 0.957
  name_count: 93
  assets_accuracy: 0.140
  assets_count: 93
  founded_accuracy: 0.548
  founded_count: 93

Overall Accuracy: 41.2%


In [68]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set_final.xml', name='fusion_test_set', nested_handling='aggregate')
# rename keypeople_name to founders for evaluation
fusion_test_set['founders'] = fusion_test_set['keypeople_name'].apply(lambda x: [x] if isinstance(x, str) else x)

# convert scientific number notation into integers
def convert_scientific_notation(value):
    try:
        if isinstance(value, str) and ('e' in value or 'E' in value):
            return int(float(value))
        return value
    except:
        return value

fusion_test_set['assets'] = fusion_test_set['assets'].apply(convert_scientific_notation)
fusion_test_set['revenue'] = fusion_test_set['revenue'].apply(convert_scientific_notation) 

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='forbes_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/companies/output/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[WARNING] PyDI.fusion.evaluation - Missing 8 expected/reference records in fused dataset: http://www.forbes.com/companies/actavis/, http://www.forbes.com/companies/petronas-dagangan/, http://www.forbes.com/companies/mondi/, http://www.forbes.com/companies/eads/, http://www.forbes.com/companies/meadwestvaco/, ...


Evaluating fusion results against gold standard...


[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.447 overall accuracy (284/635)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 351 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	revenue                          |      76 |     21.65%%
[INFO ] PyDI.fusion.evaluation - 	assets                           |      73 |     20.80%%
[INFO ] PyDI.fusion.evaluation - 	founders                         |      72 |     20.51%%
[INFO ] PyDI.fusion.evaluation - 	city                             |      50 |     14.25%%
[INFO ] PyDI.fusion.evaluation - 	founded                          |      38 |     10.83%%
[INFO ] PyDI.fusion.evaluation - 	country                          |      34 |      9.69%%
[INFO ] PyDI.fusion.evaluation - 	name                             |       8 |      2.28%%



Fusion Evaluation Results:
  overall_accuracy: 0.447
  macro_accuracy: 0.443
  num_evaluated_records: 92
  num_evaluated_attributes: 7
  total_evaluations: 635
  total_correct: 284
  city_accuracy: 0.457
  city_count: 92
  founders_accuracy: 0.133
  founders_count: 83
  country_accuracy: 0.630
  country_count: 92
  revenue_accuracy: 0.174
  revenue_count: 92
  name_accuracy: 0.913
  name_count: 92
  assets_accuracy: 0.207
  assets_count: 92
  founded_accuracy: 0.587
  founded_count: 92

Overall Accuracy: 44.7%
